# 8. Visualización y análisis de errores

Se analizan los errores de los dos modelos finales sobre el conjunto de prueba de I de 2026. Se usan las predicciones que guardó la act. 7, así que ambos modelos se comparan sobre exactamente los mismos registros.

**Definición del residuo.** En todo el análisis:

$$\text{residuo} = \text{salario real} - \text{salario predicho}$$

- **Residuo positivo:** el modelo predijo menos de lo real → **subestimación**.
- **Residuo negativo:** el modelo predijo más de lo real → **sobreestimación**.

El **error medio** de un grupo es el promedio de sus residuos (indica si el modelo se desvía hacia arriba o hacia abajo) y el **MAE** es el promedio de sus valores absolutos.

Las gráficas usan una misma muestra de 5,000 registros para los dos modelos. **Todas las métricas y tablas se calculan con los 13,258 registros de prueba.**

> Antes de correr este notebook hay que correr `ej7_evaluacion_final.ipynb`, que genera `predicciones_2026`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab7_AnalisisErrores")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)
assert spark.version.startswith("3.5."), "El laboratorio requiere Spark 3.5.x."

# Colores fijos por modelo en todas las gráficas
COLOR = {"lr": "#2a78d6", "rf": "#eb6834"}
GRIS = "#6b6a66"
NOMBRE = {"lr": "Regresión lineal", "rf": "Random Forest"}
MODELOS = ["lr", "rf"]
OBJETIVO = "salario_mensual"

formato_q = mticker.FuncFormatter(lambda x, _: f"Q{x:,.0f}")
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

## Predicciones de prueba y residuos

In [ ]:
predicciones = spark.read.parquet("../working_dir/parquet/predicciones_2026")

for m in MODELOS:
    predicciones = (predicciones
        .withColumn(f"residuo_{m}", F.col(OBJETIVO) - F.col(f"pred_{m}"))
        .withColumn(f"abs_{m}", F.abs(F.col(f"residuo_{m}"))))
predicciones = predicciones.cache()

n_prueba = predicciones.count()
print(f"Registros de prueba: {n_prueba:,}")

resumen_global = pd.DataFrame([
    {
        "modelo": NOMBRE[m],
        "registros": n_prueba,
        **predicciones.agg(
            F.avg(f"abs_{m}").alias("MAE"),
            F.sqrt(F.avg(F.col(f"residuo_{m}") ** 2)).alias("RMSE"),
            F.avg(f"residuo_{m}").alias("error_medio"),
            F.expr(f"percentile(residuo_{m}, 0.5)").alias("residuo_mediano"),
            F.avg((F.col(f"residuo_{m}") > 0).cast("int")).alias("prop_subestimados"),
            F.min(f"pred_{m}").alias("prediccion_min"),
            F.max(f"pred_{m}").alias("prediccion_max"),
            F.sum((F.col(f"pred_{m}") <= 0).cast("int")).alias("predicciones_no_positivas"),
        ).first().asDict(),
    }
    for m in MODELOS
])
display(resumen_global.round(3))

## Distribución del salario en la prueba

Antes de ver los errores, conviene recordar cómo se distribuye el salario real, porque los errores grandes se concentran donde la distribución es más dispersa.

In [ ]:
PERCENTILES = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
valores_p = predicciones.select(
    F.expr(f"percentile({OBJETIVO}, array({','.join(map(str, PERCENTILES))}))")
).first()[0]
p = dict(zip(PERCENTILES, valores_p))

estadisticos = predicciones.agg(
    F.avg(OBJETIVO).alias("media"), F.stddev(OBJETIVO).alias("desv_std"),
    F.min(OBJETIVO).alias("minimo"), F.max(OBJETIVO).alias("maximo"),
).first().asDict()

tabla_distribucion = pd.DataFrame(
    [(f"P{int(q * 100)}", v) for q, v in p.items()]
    + [("Media", estadisticos["media"]), ("Desv. estándar", estadisticos["desv_std"]),
       ("Mínimo", estadisticos["minimo"]), ("Máximo", estadisticos["maximo"])],
    columns=["estadístico", "salario_Q"])
display(tabla_distribucion.round(2))

# Histograma: conteos calculados en Spark sobre todos los registros, en escala logarítmica
bordes = np.logspace(np.log10(estadisticos["minimo"]), np.log10(estadisticos["maximo"]) + 1e-9, 41)
bucket = F.floor((F.log10(OBJETIVO) - np.log10(bordes[0])) / (np.log10(bordes[1]) - np.log10(bordes[0])))
conteos = (predicciones.groupBy(bucket.alias("bin")).count().toPandas()
           .set_index("bin")["count"].reindex(range(40), fill_value=0))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(bordes[:-1], conteos.values, width=np.diff(bordes), align="edge",
       color=GRIS, alpha=0.55, edgecolor="white", linewidth=0.5)
alto = ax.get_ylim()[1]
for q, estilo, y_rel, lado in [(0.50, "-", 0.97, "right"), (0.90, "--", 0.97, "left"),
                               (0.95, "--", 0.75, "left"), (0.99, ":", 0.97, "left")]:
    ax.axvline(p[q], color="#0b0b0b", linestyle=estilo, linewidth=1)
    etiqueta = f"P{int(q * 100)} Q{p[q]:,.0f}"
    ax.text(p[q], alto * y_rel, f" {etiqueta} " , fontsize=8, va="top", ha=lado)
ax.axvline(estadisticos["media"], color=COLOR["rf"], linewidth=1.5)
ax.text(estadisticos["media"], alto * 0.55, f" media Q{estadisticos['media']:,.0f}",
        fontsize=8, color="#0b0b0b", va="top", ha="left",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1))
ax.set_xscale("log")
ax.xaxis.set_major_formatter(formato_q)
ax.set_xlabel("Salario mensual real (escala logarítmica)")
ax.set_ylabel("Registros")
ax.set_title("Distribución del salario real en I de 2026 (todos los registros de prueba)")
plt.tight_layout()
plt.show()

**Interpretación.** El salario de prueba es asimétrico a la derecha: la mediana es Q3,200 y la media Q3,566, la mitad de las personas gana entre Q2,000 y Q4,080, pero el P95 llega a Q8,000, el P99 a Q15,000 y el máximo a Q60,000. La desviación estándar (Q2,868) es casi tan grande como la media. Además, los salarios se concentran en montos redondos; solo Q4,000 aparece 926 veces. Esta forma es clave para entender los errores: la mayor parte de los registros está en un rango estrecho y relativamente fácil de predecir, mientras que la cola derecha tiene pocos registros muy dispersos.

## Muestra común para las gráficas

Se toma **una sola muestra aleatoria de 5,000 registros** (semilla fija), y la misma muestra se usa para dibujar los dos modelos. Así, las diferencias entre gráficas se deben a los modelos y no a los puntos elegidos.

In [ ]:
TAM_MUESTRA = 5000
SEMILLA = 42

fraccion = min(1.0, 1.2 * TAM_MUESTRA / n_prueba)
muestra = (predicciones
           .sample(withReplacement=False, fraction=fraccion, seed=SEMILLA)
           .orderBy(F.rand(SEMILLA))
           .limit(TAM_MUESTRA)
           .select(OBJETIVO, *[f"pred_{m}" for m in MODELOS], *[f"residuo_{m}" for m in MODELOS])
           .toPandas())

print(f"Registros en la muestra: {len(muestra):,}")
print(f"Salario mediano: muestra Q{muestra[OBJETIVO].median():,.0f} | prueba completa Q{p[0.5]:,.0f}")

## 8.1 Salario real frente a salario predicho

Si las predicciones fueran perfectas, todos los puntos estarían sobre la línea $y = x$. Arriba se muestra la escala original en quetzales y abajo la misma gráfica en **escala logarítmica** en ambos ejes, que permite ver mejor la zona donde se concentra la mayoría de los salarios.

In [ ]:
limite = max(muestra[OBJETIVO].max(), muestra[["pred_lr", "pred_rf"]].max().max()) * 1.05
minimo_log = max(50, min(muestra[OBJETIVO].min(), muestra[["pred_lr", "pred_rf"]].min().min()) * 0.9)

fig, ejes = plt.subplots(2, 2, figsize=(12, 11))
for fila, escala in enumerate(["lineal", "log"]):
    for col, m in enumerate(MODELOS):
        ax = ejes[fila, col]
        ax.scatter(muestra[f"pred_{m}"], muestra[OBJETIVO], s=8, alpha=0.3,
                   color=COLOR[m], edgecolors="none")
        if escala == "lineal":
            ax.plot([0, limite], [0, limite], color=GRIS, linewidth=1.2, linestyle="--", label="y = x")
            ax.set_xlim(0, limite)
            ax.set_ylim(0, limite)
        else:
            ax.plot([minimo_log, limite], [minimo_log, limite], color=GRIS, linewidth=1.2,
                    linestyle="--", label="y = x")
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlim(minimo_log, limite)
            ax.set_ylim(minimo_log, limite)
        ax.xaxis.set_major_formatter(formato_q)
        ax.yaxis.set_major_formatter(formato_q)
        ax.tick_params(axis="x", labelrotation=30)
        ax.set_xlabel("Salario predicho" + (" (escala log)" if escala == "log" else ""))
        ax.set_ylabel("Salario real" + (" (escala log)" if escala == "log" else ""))
        titulo = f"{NOMBRE[m]} — escala {'logarítmica' if escala == 'log' else 'original'}"
        if escala == "log":
            no_positivas = (muestra[f"pred_{m}"] <= 0).sum()
            if no_positivas:
                titulo += f"\n({no_positivas} predicciones ≤ Q0 no se pueden dibujar en escala log)"
        ax.set_title(titulo)
        ax.legend(loc="upper left")
        ax.set_aspect("equal")

fig.suptitle("Salario real vs. predicho en I de 2026 (misma muestra de 5,000 registros)", fontsize=13)
plt.tight_layout()
plt.show()

**Interpretación.**

- **En la zona central, los dos modelos siguen la línea y = x**, pero con mucha dispersión vertical: para una misma predicción, el salario real varía mucho. Las **bandas horizontales** que se ven en escala logarítmica son los salarios redondos (Q2,000, Q3,000, Q4,000…), que reciben predicciones muy distintas según las características de cada persona.
- **Los salarios altos quedan por encima de la línea**: hay personas que ganan Q20,000–Q50,000 con predicciones de apenas Q5,000–Q15,000. Ningún modelo predice valores cercanos a los salarios más altos. La regresión lineal nunca pasa de Q15,663, y el Random Forest llega como máximo a Q35,097, aunque solo en casos muy puntuales.
- **La regresión lineal forma columnas separadas** en la escala original (por ejemplo, un grupo de predicciones cerca de Q11,000). Esto ocurre porque suma efectos fijos por categoría: todas las personas con la misma combinación de educación, categoría ocupacional y dominio reciben casi la misma predicción. El Random Forest produce predicciones más continuas porque combina las variables de forma no lineal.
- **La regresión lineal produce predicciones imposibles:** 29 registros de prueba reciben un salario predicho negativo (el mínimo es −Q496; 12 de ellos están en la muestra y no aparecen en la escala logarítmica), y abajo a la izquierda sobreestima los salarios muy bajos. El Random Forest no puede predecir fuera del rango de salarios del entrenamiento, así que su mínimo es Q468.

## 8.2 Residuos frente al salario predicho

Cada punto es un registro de la muestra; la línea horizontal marca el residuo cero. Además se dibuja, **con todos los registros de prueba**, el residuo medio por decil del salario predicho, para ver la tendencia sin depender de la muestra.

In [ ]:
def residuo_por_decil_prediccion(m):
    cortes = predicciones.approxQuantile(f"pred_{m}", [i / 10 for i in range(1, 10)], 0.0)
    decil = F.lit(0)
    for c in cortes:
        decil = decil + (F.col(f"pred_{m}") > c).cast("int")
    return (predicciones.groupBy(decil.alias("decil"))
            .agg(F.avg(f"pred_{m}").alias("pred_media"), F.avg(f"residuo_{m}").alias("residuo_medio"))
            .orderBy("decil").toPandas())

lim_res = np.nanpercentile(np.abs(muestra[["residuo_lr", "residuo_rf"]].values), 99.5) * 1.1
lim_pred = muestra[["pred_lr", "pred_rf"]].max().max() * 1.05

fig, ejes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
for ax, m in zip(ejes, MODELOS):
    ax.scatter(muestra[f"pred_{m}"], muestra[f"residuo_{m}"], s=8, alpha=0.3,
               color=COLOR[m], edgecolors="none", label="registro (muestra)")
    ax.axhline(0, color="#0b0b0b", linewidth=1.2)
    tendencia = residuo_por_decil_prediccion(m)
    ax.plot(tendencia["pred_media"], tendencia["residuo_medio"], color="#0b0b0b", marker="o",
            markersize=5, linewidth=2, label="residuo medio por decil (todos)")
    fuera = (muestra[f"residuo_{m}"].abs() > lim_res).sum()
    ax.set_ylim(-lim_res, lim_res)
    ax.set_xlim(0, lim_pred)
    ax.xaxis.set_major_formatter(formato_q)
    ax.yaxis.set_major_formatter(formato_q)
    ax.set_xlabel("Salario predicho")
    ax.set_title(f"{NOMBRE[m]}\n({fuera} puntos con |residuo| > Q{lim_res:,.0f} fuera del eje)")
    ax.legend(loc="upper right")
    ax.text(0.01, 0.98, "↑ subestima", transform=ax.transAxes, va="top", fontsize=9, color=GRIS)
    ax.text(0.01, 0.02, "↓ sobreestima", transform=ax.transAxes, va="bottom", fontsize=9, color=GRIS)
ejes[0].set_ylabel("Residuo = real − predicho")
fig.suptitle("Residuos vs. salario predicho en I de 2026 (misma muestra de 5,000 registros)", fontsize=13)
plt.tight_layout()
plt.show()

**Interpretación.**

- **En promedio, los errores están centrados cerca de cero** en casi todo el rango de predicciones: la línea negra, calculada con todos los registros por decil de predicción, se mantiene muy cerca de 0. Es decir, dada una predicción, el modelo no se equivoca sistemáticamente hacia arriba ni hacia abajo. Solo en el decil de predicciones más altas el residuo medio sube (≈ Q500–Q700), lo que indica una leve subestimación en ese extremo.
- **Los errores no tienen varianza constante (heterocedasticidad):** la nube se abre a medida que crece el salario predicho. Para predicciones de Q2,000 los residuos están casi todos entre −Q2,000 y +Q3,000, mientras que para predicciones de Q10,000 van de −Q10,000 a +Q10,000 o más.
- **La nube es asimétrica:** hacia abajo el residuo está limitado (el peor caso es predecir de más a alguien que gana muy poco), pero hacia arriba hay residuos muy grandes, de salarios altos que el modelo no anticipa. Por eso el residuo **mediano** es negativo (−Q31 en regresión lineal y −Q57 en Random Forest) mientras que el **error medio** es positivo (+Q121 y +Q106): la mayoría de las personas recibe una predicción un poco alta, y unas pocas con salarios altos quedan muy subestimadas.
- Las **líneas diagonales** paralelas que se ven en la nube son de nuevo los salarios redondos: para un salario real fijo, el residuo baja una unidad por cada unidad que sube la predicción.
- El Random Forest tiene la nube un poco más compacta, lo que es coherente con su menor MAE y RMSE.

## 8.3 Error por nivel educativo y por dominio

In [ ]:
def tabla_por_grupo(codigo, etiqueta):
    agregados = [F.count("*").alias("n"), F.avg(OBJETIVO).alias("salario_real_medio")]
    for m in MODELOS:
        agregados += [F.avg(f"abs_{m}").alias(f"MAE_{m}"), F.avg(f"residuo_{m}").alias(f"error_medio_{m}")]
    return (predicciones.groupBy(codigo, etiqueta).agg(*agregados)
            .orderBy(codigo).toPandas()
            .rename(columns={etiqueta: "grupo"}).drop(columns=codigo))

tabla_educacion = tabla_por_grupo("nivel_educativo", "nivel_educativo_etiqueta")
tabla_dominio = tabla_por_grupo("dominio", "dominio_etiqueta")

print("Por nivel educativo")
display(tabla_educacion.round(2))
print("Por dominio")
display(tabla_dominio.round(2))

In [ ]:
def grafica_error_grupos(tabla, titulo, ax, loc_leyenda):
    y = np.arange(len(tabla))
    alto = 0.38
    for i, m in enumerate(MODELOS):
        ax.barh(y + (i - 0.5) * alto, tabla[f"error_medio_{m}"], height=alto - 0.04,
                color=COLOR[m], label=NOMBRE[m])
    ax.axvline(0, color="#0b0b0b", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{g} (n={n:,})" for g, n in zip(tabla["grupo"], tabla["n"])])
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(formato_q)
    ax.set_xlabel("Error medio (real − predicho)   ←  sobreestima | subestima  →")
    ax.set_title(titulo)
    ax.legend(loc=loc_leyenda)

fig, ejes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.3, 1]})
grafica_error_grupos(tabla_educacion, "Error medio por nivel educativo", ejes[0], "center right")
grafica_error_grupos(tabla_dominio, "Error medio por dominio", ejes[1], "lower right")
plt.tight_layout()
plt.show()

**Interpretación.**

**Por nivel educativo:**
- **El MAE crece con el nivel educativo**, al igual que el salario: con el Random Forest va de ≈ Q630–Q760 en los niveles bajos (ninguno, preprimaria, primaria) a Q2,291 en superior, Q4,680 en maestría y Q10,996 en doctorado. En términos relativos al salario promedio del grupo, el error también es mayor en los niveles altos (≈ 37 % en superior contra ≈ 31 % en primaria), porque en esos grupos los salarios son mucho más dispersos.
- **El error medio es pequeño en los grupos grandes** (primaria, básico y diversificado, entre +Q45 y +Q131), así que ahí los modelos casi no se desvían hacia un lado. En **superior** ambos modelos **subestiman** en ≈ Q260–Q300 en promedio.
- **Doctorado** tiene el mayor error (MAE ≈ Q11,000 y subestimación media ≈ Q6,000), pero solo tiene **12 registros**, así que ese resultado es muy inestable y no permite conclusiones firmes. En **maestría** (183 registros) los modelos sobreestiman levemente.
- El Random Forest tiene menor MAE que la regresión lineal en **todos** los niveles educativos. La mayor ventaja relativa está en los niveles bajos (por ejemplo, en "ninguno" baja de Q824 a Q653), donde la regresión lineal da algunas predicciones muy bajas o negativas.

**Por dominio:**
- El MAE sigue el nivel salarial de cada dominio: es mayor en **Urbano Metropolitano** (Q1,319 con Random Forest), intermedio en **Resto Urbano** (Q1,044) y menor en **Rural Nacional** (Q773). En Urbano Metropolitano el salario promedio es más alto y más disperso.
- En los tres dominios el error medio es **positivo**, es decir, una leve **subestimación**, más marcada en Urbano Metropolitano (≈ +Q136 a +Q161) que en el área rural (+Q17 a +Q65). Aun así, son sesgos pequeños frente al MAE: los errores se deben sobre todo a dispersión, no a un sesgo sistemático por dominio.
- El Random Forest mejora el MAE en los tres dominios.

## 8.4 Errores según el percentil del salario real

In [ ]:
TRAMOS = [(0.00, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 0.90), (0.90, 0.95), (0.95, 0.99), (0.99, 1.00)]
cortes = {q: p[q] for q in [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]}

tramo = F.lit(f"P0–P25")
for (inf, sup) in TRAMOS[1:]:
    tramo = F.when(F.col(OBJETIVO) > cortes[inf], F.lit(f"P{int(inf * 100)}–P{int(sup * 100)}")).otherwise(tramo)
orden = {f"P{int(i * 100)}–P{int(s * 100)}": k for k, (i, s) in enumerate(TRAMOS)}

agregados = [F.count("*").alias("n"), F.min(OBJETIVO).alias("salario_min"),
             F.max(OBJETIVO).alias("salario_max"), F.avg(OBJETIVO).alias("salario_real_medio")]
for m in MODELOS:
    agregados += [F.avg(f"pred_{m}").alias(f"pred_media_{m}"),
                  F.avg(f"residuo_{m}").alias(f"error_medio_{m}"),
                  F.avg(f"abs_{m}").alias(f"MAE_{m}"),
                  F.sum(F.col(f"residuo_{m}") ** 2).alias(f"sse_{m}")]

tabla_tramos = predicciones.groupBy(tramo.alias("tramo")).agg(*agregados).toPandas()
tabla_tramos = tabla_tramos.sort_values("tramo", key=lambda s: s.map(orden)).reset_index(drop=True)
for m in MODELOS:
    tabla_tramos[f"pct_error_cuadratico_{m}"] = 100 * tabla_tramos[f"sse_{m}"] / tabla_tramos[f"sse_{m}"].sum()
    tabla_tramos[f"pct_subestimado_{m}"] = np.nan
tabla_tramos["pct_registros"] = 100 * tabla_tramos["n"] / n_prueba

sub = (predicciones.groupBy(tramo.alias("tramo"))
       .agg(*[(100 * F.avg((F.col(f"residuo_{m}") > 0).cast("int"))).alias(f"pct_subestimado_{m}") for m in MODELOS])
       .toPandas().set_index("tramo"))
for m in MODELOS:
    tabla_tramos[f"pct_subestimado_{m}"] = tabla_tramos["tramo"].map(sub[f"pct_subestimado_{m}"])

columnas = ["tramo", "n", "pct_registros", "salario_min", "salario_max", "salario_real_medio"]
for m in MODELOS:
    columnas += [f"pred_media_{m}", f"error_medio_{m}", f"MAE_{m}", f"pct_subestimado_{m}", f"pct_error_cuadratico_{m}"]
display(tabla_tramos[columnas].round(1))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(tabla_tramos))
ancho = 0.38

for i, m in enumerate(MODELOS):
    ejes[0].bar(x + (i - 0.5) * ancho, tabla_tramos[f"error_medio_{m}"], width=ancho - 0.04,
                color=COLOR[m], label=NOMBRE[m])
ejes[0].axhline(0, color="#0b0b0b", linewidth=1)
ejes[0].set_title("Error medio por tramo del salario real")
ejes[0].set_ylabel("Error medio (real − predicho)\n↓ sobreestima | subestima ↑")
ejes[0].yaxis.set_major_formatter(formato_q)

for i, m in enumerate(MODELOS):
    ejes[1].bar(x + (i - 0.5) * ancho, tabla_tramos[f"pct_error_cuadratico_{m}"], width=ancho - 0.04,
                color=COLOR[m], label=NOMBRE[m])
ejes[1].plot(x, tabla_tramos["pct_registros"], color="#0b0b0b", marker="o", markersize=5,
             linewidth=1.5, label="% de registros")
ejes[1].set_title("Aporte al error cuadrático total (RMSE) vs. % de registros")
ejes[1].set_ylabel("% del total")

for ax in ejes:
    ax.set_xticks(x)
    ax.set_xticklabels(tabla_tramos["tramo"])
    ax.set_xlabel("Tramo de percentil del salario real")
    ax.legend()
plt.tight_layout()
plt.show()

**Interpretación: ¿los modelos subestiman o sobreestiman los salarios altos?**

Los dos modelos **subestiman sistemáticamente los salarios altos y sobreestiman los bajos.** Es el efecto típico de *regresión hacia la media*: con seis predictores, los modelos no pueden distinguir bien a las personas que ganan mucho de las que tienen un perfil parecido pero ganan lo normal, así que "encogen" las predicciones hacia el centro de la distribución.

- **Salarios bajos (hasta P25, ≤ Q2,000):** error medio de −Q829 (regresión lineal) y −Q765 (Random Forest). Los modelos **sobreestiman** en promedio: predicen ≈ Q2,100 a personas que ganan en promedio Q1,298.
- **Zona central (P25–P75):** el error medio es casi cero (entre −Q48 y −Q163) y cerca de la mitad de los registros queda subestimada y la otra mitad sobreestimada. Aquí es donde los modelos funcionan mejor.
- **Salarios altos:** desde P75 el error medio se vuelve positivo y crece rápido. Entre P90 y P95 los modelos subestiman en ≈ Q1,500–Q1,600 y lo hacen en el 84–90 % de los casos. Entre P95 y P99 la subestimación media es de ≈ Q3,500–Q4,100 (93 % de los casos). **En el 1 % más alto (salarios de Q15,088 a Q60,000) el 100 % de los registros queda subestimado**, en ≈ Q12,500 (Random Forest) y ≈ Q14,300 (regresión lineal) en promedio.
- **Los salarios altos dominan el RMSE:** el 5 % de registros con salarios por encima del P95 aporta **≈ 59–62 % del error cuadrático total** (20.6 % + 41.1 % en la regresión lineal; 21.0 % + 38.3 % en el Random Forest). Solo el 1 % más alto aporta ≈ 40 %. Por eso el RMSE (≈ Q2,000) es mucho mayor que el MAE (≈ Q1,100): unos pocos errores enormes pesan mucho más que los muchos errores moderados.
- **El Random Forest reduce la subestimación en la cola alta** (en el 1 % más alto predice en promedio Q10,214 contra Q8,445 de la regresión lineal), porque sus árboles pueden aislar combinaciones de características asociadas a salarios altos. Aun así, está lejos de corregirla.

(El tramo P0–P25 tiene 30 % de los registros, y no 25 %, porque muchos salarios son exactamente Q2,000, que es el P25, y todos esos empates quedan en el mismo tramo.)

## Discusión final

**1. Datos y población analizada.** Al unir los cuatro trimestres de 2025 (203,676 registros) y aplicar los filtros de población (15 años o más, ocupados, asalariados con salario positivo) quedan 53,025 registros en 2025 y 13,258 en I de 2026, ≈ 26 % en cada archivo. Todos los faltantes de las variables usadas resultaron estructurales (preguntas que no aplican), la clave persona–período es única y todos los códigos coinciden con el diccionario, así que los filtros de calidad no eliminaron registros. Los resultados describen a **asalariados con salario registrado en la muestra**, sin ponderar: no son estimaciones oficiales para Guatemala. Además, una misma persona puede aparecer en varios trimestres por el diseño con rotación de la encuesta.

**2. El salario es muy desigual y asimétrico.** La mayoría de los salarios se concentra entre Q2,000 y Q4,000, con muchos valores redondos, y hay una cola larga de salarios altos que separa claramente la media de la mediana. El salario mediano crece con el nivel educativo y es mayor en el empleo de gobierno y en el dominio urbano metropolitano. Las correlaciones lineales entre el salario y las variables numéricas (edad, antigüedad, horas) son débiles, lo que ya anticipaba que la parte numérica de los predictores explica poco por sí sola.

**3. Perfiles de trabajadores (KMeans).** Con edad, antigüedad y horas estandarizadas se eligió K = 4. Los perfiles resultantes son: jóvenes con poca antigüedad (el grupo más grande), veteranos con mucha antigüedad y más presencia en el sector público (el de salario mediano más alto), trabajadores con jornadas muy largas y adultos con reingreso reciente. Incluir el salario empeoró la segmentación, lo que refleja que el salario no se separa bien en grupos a partir de estas variables. Tres de los cuatro perfiles tienen el mismo salario mediano (Q3,000) a pesar de perfiles muy distintos, lo que es otra señal de que el salario depende de factores que no están en estas variables.

**4. ¿Qué tan bien se puede estimar el salario?** Con los seis predictores, el mejor modelo (Random Forest) explica ≈ 53 % de la variación del salario en 2026, con un error absoluto medio de ≈ Q1,100, frente a ≈ 43 % y Q1,241 de la regresión lineal. Los dos superan claramente a predecir siempre la media (R² ≈ 0), y su desempeño en 2026 es casi igual al de la validación, así que generalizan bien a un período nuevo. El Random Forest es mejor porque captura relaciones no lineales e interacciones entre educación, categoría ocupacional, dominio, edad y horas, y además nunca da predicciones negativas, a diferencia de la regresión lineal.

**5. Dónde fallan los modelos.** El error no es uniforme:
- Crece con el nivel salarial (heterocedasticidad): es mayor en educación superior o de posgrado y en el área urbana metropolitana.
- Los modelos **sobreestiman los salarios bajos y subestiman los altos**. En el 1 % más alto, todos los registros quedan subestimados, en más de Q12,000 en promedio.
- Ese 5 % de salarios más altos concentra ≈ 60 % del error cuadrático, así que el RMSE depende sobre todo de unos pocos casos extremos, que se conservaron en la evaluación como pedía el enunciado.